## Prepare requierments

In [ ]:
pip uninstall -y transformers


In [ ]:
!pip install trl

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers
!pip install peft
!pip install accelerate datasets bitsandbytes

## import Libraries

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
import torch
import os
from peft import LoraConfig, get_peft_model

In [ ]:
print("CUDA:", torch.cuda.is_available())
print("TRL OK")

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_properties(0).total_memory/1e9)

## load Dataset

In [ ]:
from datasets import load_dataset

data = load_dataset("sahil2801/CodeAlpaca-20k", split="train")
len(data)

In [ ]:
dataset = data

In [ ]:
for i in range(2): # 3
  for k, v in dataset[i].items():
    print(k, ":", v)
  print("-----"*10)

In [ ]:
keywords = [
    # Libraries
    "matplotlib", "seaborn", "pandas", "numpy",

    # Visualization
    "plot", "visualize", "graph", "chart",
    "histogram", "bar", "line", "scatter",
    "distribution", "boxplot", "heatmap",

    # Data terms
    "dataframe", "dataset", "csv",
    "analyze", "analysis", "statistics",
    "mean", "median", "correlation"
]

In [ ]:
def filter_visualization(example):
    text = (
        example["instruction"] +
        example["input"] +
        example["output"]
    ).lower()

    return any(keyword in text for keyword in keywords)

In [ ]:
dataset = dataset.filter(filter_visualization)

In [ ]:
import pandas as pd

df = dataset.to_pandas()
print(df.isna().sum())

n = len(dataset)
print(f"Number of samples after filter : {n}")

In [ ]:
def preprocess_function(example):

    if example["input"].strip() != "":
        user_content = (
            f"{example['instruction']}\n\n"
            f"{example['input']}"
        )
    else:
        user_content = example["instruction"]

    return {
        "prompt": [
            {
                "role": "user",
                "content": user_content,
            }
        ],
        "completion": [
            {
                "role": "assistant",
                "content": example["output"],
            }
        ],
    }


In [ ]:
def preprocess_function(example):
    if example["input"].strip() != "":
        user_content = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}"
        )
    else:
        user_content = f"### Instruction:\n{example['instruction']}"

    return {
        "text": (
            f"{user_content}\n\n"
            f"### Response:\n{example['output']}"
        )
    }

In [ ]:
dataset = dataset.map(
    preprocess_function,
    remove_columns=dataset.column_names)

In [ ]:
dataset = dataset.select(range(800))

In [ ]:
for i in range(2): # 3

    for k, v in dataset[i].items():
        print(k, ":", v)
        print("-----"*3)
    print("===="*20)

## Get model

In [ ]:
model_name = "HuggingFaceTB/SmolLM3-3B"

model = AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name,
                                             device_map="auto",
                                             torch_dtype=torch.float16)

tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

## Training config

In [ ]:
# Load model with PEFT config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",

    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj"
    ]
)

In [ ]:
config = SFTConfig(
    output_dir="./smollm3-visualization-sft",
    per_device_train_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    fp16=True,
    dataset_text_field="text"
)


In [ ]:
import trl
print(trl.__version__)

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=config,
)

## Training

In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
trainer.train()

## Save New model - Push to Hugging face hub

In [ ]:
trainer.save_model("./smollm3-visualization-sft/final")
tokenizer.save_pretrained("./smollm3-visualization-sft/final")
print("Model saved!")

In [ ]:
trainer.model.save_pretrained("lora_adapter")
tokenizer.save_pretrained("lora_adapter")

In [ ]:
from google.colab import userdata
HF_token = userdata.get('Hugging_face_tokens')

In [ ]:
from huggingface_hub import HfApi
import os
api = HfApi(token= HF_token)


In [ ]:
from huggingface_hub import login
login(token= HF_token)

In [ ]:
from huggingface_hub import whoami
whoami()

In [ ]:
from huggingface_hub import HfApi
api = HfApi()

api.create_repo(
    repo_id="Kyrollos32/smollm_sft_2",
    repo_type="model",
    exist_ok=True
)

In [ ]:
api.upload_folder(
    folder_path="lora_adapter",
    repo_id="Kyrollos32/smollm_sft_2",
    repo_type="model",
)

In [ ]:
trainer.push_to_hub("Kyrollos32/smollm3-CodeGenerator")
tokenizer.push_to_hub("Kyrollos32/smollm3-CodeGenerator")


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM3-3B",
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM3-3B")

model = PeftModel.from_pretrained(
    base_model,
    "Kyrollos32/smollm_sft_2"
)

In [ ]:
model.eval()

In [ ]:
prompt = """
Write Python code using matplotlib to create a line plot
for a DataFrame column named 'sales'.
"""

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.2,
        top_p=0.9
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

In [ ]:
response = response[len(prompt):]
print(response)

In [ ]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

result = pipe(prompt, max_new_tokens=200)
print(result[0]["generated_text"])

# Merge and push

In [ ]:
merged_model = model.merge_and_unload()

In [ ]:
merged_model.save_pretrained("smollm_sft_2_full")
tokenizer.save_pretrained("smollm_sft_2_full")

In [ ]:
api.create_repo(
    repo_id="Kyrollos32/smollm_sft_2_full",
    repo_type="model",
    exist_ok=True
)

In [ ]:
api.upload_folder(
    folder_path="smollm_sft_2_full",
    repo_id="Kyrollos32/smollm_sft_2_full",
    repo_type="model",
)

In [ ]:
# How to use it

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "Kyrollos32/smollm_sft_2_full",
    torch_dtype=torch.float16,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(
    "Kyrollos32/smollm_sft_2_full"
)

model.eval()